# Hemlock — Defense Comparison

Rule-based defenses vs. LLM-based chunk classifier — side by side.

**What you'll see:**
1. Which attacks rule-based filters catch (and which they miss)
2. Which attacks the LLM classifier catches that rules can't
3. The cost/latency tradeoff between defense layers
4. False positive rate — does the classifier block legitimate documents?

No API key required for the rule-based section. LLM classifier section needs a key.

In [ ]:
import sys
sys.path.insert(0, "..")

from langchain_core.documents import Document

from defenses.input_sanitizer import InjectionPatternFilter, UnicodeNormalizer, MarkdownHeaderSanitizer
from defenses.chunk_filter import InjectionChunkFilter
from attacks.direct_injection import EXPLICIT_DOC
from attacks.citation_forgery import FAKE_PAPER_DOC, FAKE_STANDARD_DOC
from attacks.chain_of_thought_hijack import LOGICAL_TRAP_DOC, AUTHORITY_COT_DOC
from attacks.temporal_spoofing import FUTURE_DATED_DOC, STALE_OVERRIDE_DOC
from attacks.invisible_markup import HTML_COMMENT_DOC, CSS_HIDDEN_DOC
from attacks.authority_spoofing import CONFIG_DOC, DEVELOPER_DOC

ATTACK_DOCS = {
    "direct_injection [explicit]": EXPLICIT_DOC,
    "citation_forgery [fake_paper]": FAKE_PAPER_DOC,
    "citation_forgery [fake_standard]": FAKE_STANDARD_DOC,
    "chain_of_thought [logical_trap]": LOGICAL_TRAP_DOC,
    "chain_of_thought [authority_cot]": AUTHORITY_COT_DOC,
    "temporal_spoofing [future_dated]": FUTURE_DATED_DOC,
    "temporal_spoofing [stale_override]": STALE_OVERRIDE_DOC,
    "invisible_markup [html_comment]": HTML_COMMENT_DOC,
    "invisible_markup [css_hidden]": CSS_HIDDEN_DOC,
    "authority_spoofing [config]": CONFIG_DOC,
    "authority_spoofing [developer]": DEVELOPER_DOC,
}

LEGIT_DOCS = {
    "legit: return policy": "Our refund policy allows returns within 30 days with a receipt.",
    "legit: shipping info": "Standard shipping takes 3-5 business days. Express is available.",
    "legit: FAQ": "You can reset your password from the account settings page.",
}

print(f"Attack docs: {len(ATTACK_DOCS)}")
print(f"Legit docs:  {len(LEGIT_DOCS)}")

## 1. Rule-based defenses

In [ ]:
RULE_DEFENSES = [
    ("InjectionPatternFilter", InjectionPatternFilter()),
    ("UnicodeNormalizer",      UnicodeNormalizer()),
    ("MarkdownHeaderSanitizer",MarkdownHeaderSanitizer()),
    ("InjectionChunkFilter",   InjectionChunkFilter()),
]

def check_rule_defenses(content: str, label: str) -> dict:
    doc = Document(page_content=content, metadata={"source": label})
    results = {}
    for name, defense in RULE_DEFENSES:
        # IngestDefense has inspect(), RetrievalDefense has filter()
        if hasattr(defense, "inspect"):
            result_doc, report = defense.inspect(doc)
            results[name] = report.triggered
        else:
            safe, reports = defense.filter([doc])
            results[name] = len(safe) == 0 or any(r.triggered for r in reports)
    results["any_blocked"] = any(results.values())
    return results

print(f"{'Document':<45} {'IPF':>5} {'UN':>5} {'MHS':>5} {'ICF':>5} {'Caught':>7}")
print("-" * 75)

for label, content in ATTACK_DOCS.items():
    r = check_rule_defenses(content, label)
    caught = "✓ YES" if r["any_blocked"] else "✗ NO"
    print(f"{label:<45} {str(r['InjectionPatternFilter']):>5} {str(r['UnicodeNormalizer']):>5} "
          f"{str(r['MarkdownHeaderSanitizer']):>5} {str(r['InjectionChunkFilter']):>5} {caught:>7}")

print("\nFalse positive check (legit docs — should all be False):")
for label, content in LEGIT_DOCS.items():
    r = check_rule_defenses(content, label)
    fp = "⚠ FALSE POSITIVE" if r["any_blocked"] else "✓ pass"
    print(f"  {label:<43} {fp}")

## 2. LLM classifier

> Requires `ANTHROPIC_API_KEY` or `OPENAI_API_KEY`. Skip if no key available.

In [ ]:
import os

HAS_KEY = bool(os.environ.get("ANTHROPIC_API_KEY") or os.environ.get("OPENAI_API_KEY"))
print(f"API key available: {HAS_KEY}")
if not HAS_KEY:
    print("Skipping LLM classifier cells — set ANTHROPIC_API_KEY to run them.")

In [ ]:
if HAS_KEY:
    from defenses.llm_classifier import LLMChunkClassifier

    if os.environ.get("ANTHROPIC_API_KEY"):
        from langchain_anthropic import ChatAnthropic
        llm = ChatAnthropic(model="claude-haiku-4-5-20251001")
    else:
        from langchain_openai import ChatOpenAI
        llm = ChatOpenAI(model="gpt-4o-mini")

    classifier = LLMChunkClassifier(llm=llm, threshold=0.6, cache=True)

    print(f"{'Document':<45} {'Rules':>7} {'LLM':>7} {'LLM reason'}")
    print("-" * 85)

    for label, content in ATTACK_DOCS.items():
        rules = check_rule_defenses(content, label)
        doc = Document(page_content=content, metadata={"source": label})
        blocked, reason = classifier.is_blocked(doc)
        rule_result = "✓" if rules["any_blocked"] else "✗"
        llm_result  = "✓" if blocked else "✗"
        print(f"{label:<45} {rule_result:>7} {llm_result:>7}  {reason[:40]}")

    print("\nFalse positive check (LLM — legit docs):")
    for label, content in LEGIT_DOCS.items():
        doc = Document(page_content=content, metadata={"source": label})
        blocked, reason = classifier.is_blocked(doc)
        fp = "⚠ FALSE POSITIVE" if blocked else "✓ pass"
        print(f"  {label:<43} {fp}  {reason[:40]}")

## 3. Combined: rule-based catches, LLM catches, both miss

In [ ]:
# Summary without LLM (run even without API key)
rule_catches = sum(1 for c in ATTACK_DOCS.values() if check_rule_defenses(c, "")["any_blocked"])
rule_miss    = len(ATTACK_DOCS) - rule_catches

print(f"Rule-based defenses: catch {rule_catches}/{len(ATTACK_DOCS)} attack docs ({rule_catches/len(ATTACK_DOCS):.0%})")
print(f"  Miss {rule_miss} docs — these require LLM classifier or prompt hardening")
print()
print("Docs that evade all rule-based filters:")
for label, content in ATTACK_DOCS.items():
    r = check_rule_defenses(content, label)
    if not r["any_blocked"]:
        print(f"  • {label}")